# TP n°8 — Réseaux de Petri

## Objectifs
Modéliser et vérifier des systèmes concurrents à l’aide des Réseaux de Petri (RdP).

## Rappel
Les RdP constituent un formalisme pour la description et l’analyse des systèmes concurrents.  
L’ensemble minimal de couvertures est une représentation finie de l’ensemble des marquages accessibles. Il peut être utilisé pour résoudre plusieurs problèmes, parmi lesquels déterminer si le réseau est borné (i.e. possède un nombre fini de marquages accessibles).

### Algorithme de Karp et Miller
L’algorithme de Karp et Miller construit un arbre étiqueté par les marquages accessibles :

1. **Initialiser** un arbre avec un sommet étiqueté par le marquage initial.
2. **Tant qu’il existe des feuilles non traitées** dans l’arbre :
   1. Prendre une feuille *f* de l’arbre, étiquetée par *M*.
   2. Si *f* a un ancêtre étiqueté par *M*, passer à la feuille suivante.
   3. Si *f* a un ancêtre étiqueté par *N* tel que  
      ∀q ∈ P, M(q) ≤ N(q) **et** ∃p ∈ P, M(p) < N(p) alors **arrêter** (le réseau est non borné).
   4. Sinon, pour chaque transition *t* admissible par *M* :
      1. Calculer *M′* tel que M [t > M′.
      2. Ajouter un fils à *f* étiqueté par *M′*.
3. Si **toutes les feuilles ont été parcourues**, alors le réseau est **borné**.


## Exercice 1 — Algorithme de Karp et Miller
Vous représentez un RdP par :
- un nombre de places **P** ;
- un ensemble de transitions **T** ;
- un tableau `pre[p][t]` indiquant combien de jetons doivent être retirés de la place *p* lors du franchissement de la transition *t* ;
- un tableau `post[p][t]` indiquant combien de jetons doivent être ajoutés dans la place *p* lors du franchissement de la transition *t*.

Comme un marquage peut apparaître dans plusieurs nœuds de l’arbre, vous devez associer un **identifiant** à chaque nœud. Représentez l’arbre comme :
- un tableau qui, à chaque identifiant, associe un **marquage** ;
- un **dictionnaire** qui, à chaque identifiant, associe son **parent** dans l’arbre. Le marquage initial est son **propre parent**.

**Question 1.** Implémentez l’algorithme de Karp et Miller.

In [1]:
# Rappels : on représente un réseau de Petri par 
## Un nombre de places P
## Un nombre de transitions T
## Un tableau pre[p][t] qui indique combien de jetons doivent être pris dans p quand on emprunte t
## Un tableau post[p][t] qui indique combien de jetons doivent être ajoutés dans p quand on emprunte t

# Un marquage est représenté par un tableau m (m[p]=nombre de jetons dans p)

def apply(P, T, pre, post, m, t):
    #renvoie le marquage obtenu en appliquant, si possible, la transition t au marquage m
    for p in range(P):
        if m[p] < pre[p][t]:
            return None  # transition non franchissable
    return [m[p] - pre[p][t] + post[p][t] for p in range(P)]

def franchissable(P, T, pre, post, m):
    # renvoie la liste des transitions franchissables à partir du marquage m
    return [t for t in range(T) if all(m[p] >= pre[p][t] for p in range(P))]

# L'algorithme de Karp et Miller construit un arbre étiqueté par les marquages accessibles. 
# Comme un marquage peut apparaître dans plusieurs noeuds de l'arbre, on va associer un identifiant à chaque noeud.
# On représente l'arbre comme :
# - un dictionnaire qui à chaque identifiant associe son parent dans l'arbre 
# - un tableau qui à chaque identifiant associe un marquage
# (l'identifiant du marquage initial sera son propre parent)

def ajout_fils(arbre_dict, arbre_tab, iden, m):
    # fonction qui ajoute m dans l'arbre, en tant que fils du noeud iden
    new_id = len(arbre_tab)
    arbre_tab.append(m)
    arbre_dict[new_id] = iden
    return (arbre_dict, arbre_tab)

def ancetres(arbre_dict, arbre_tab, iden):
    # renvoie la liste des ancêtres d'un noeud dans l'arbre
    if iden == 0: 
        return [arbre_tab[iden]]
    else: 
        return ancetres(arbre_dict, arbre_tab, arbre_dict[iden]) + [arbre_tab[iden]]

def temoin(m1, m2):
    # teste si m1 ->* m2 est un témoin de non-bornitude :
    # pour tout p, m1[p] <= m2[p]
    # il existe q, m1[q] < m2[q]
    # (donc renvoie True si "m1 est inclus dans m2", false sinon)
    return (all(m1[p] <= m2[p] for p in range(len(m1))) and
            any(m1[p] < m2[p] for p in range(len(m1))))

def estborne(P, T, pre, post, m):
    #Algo de Karp et Miller

    #Initialiser l'arbre
    arbre_dict = {0: 0}  # le marquage initial est son propre parent
    arbre_tab = [m]
    To_do = [0] 

    while To_do != []:
        iden = To_do.pop(0)
        M = arbre_tab[iden]


        if iden == 0:
            ancs = []  # la racine n'a pas d'ancêtres
        else:
            ancs = ancetres(arbre_dict, arbre_tab, arbre_dict[iden])

        if M in ancs:
            continue

        for N in ancs:
            if temoin(N, M):
                return False


        for t in franchissable(P, T, pre, post, M):
            M_prime = apply(P, T, pre, post, M, t)
            arbre_dict, arbre_tab = ajout_fils(arbre_dict, arbre_tab, iden, M_prime)
            new_id = len(arbre_tab) - 1
            To_do.append(new_id)

    return True

P3 = 2
T3 = 3
pre3 = [[1,0,0],[0,1,1]]
post3 = [[0,1,0],[1,0,0]]
m3 = [1,0]

print(estborne(P3,T3,pre3,post3,m3))

True


## Exercice 2 — Génération du graphe des marquages accessibles
L’algorithme suivant génère le graphe des marquages accessibles \((V, E)\) d’un RdP **borné** :

1. Initialiser une **pile S** avec le marquage initial.
2. **V** contient initialement le marquage initial.
3. Tant que **S** est non vide :
   1. **Dépiler** le sommet *M*.
   2. Pour chaque transition *t* admissible depuis *M*, calculer *M′* obtenu après franchissement de *t*.
   3. Si *M′* **n’est pas** dans **V**, l’ajouter et **empiler** *M′*.
   4. Ajouter l’arc **M → M′** étiqueté par *t* à **E**.

**Question 1.** Implémentez cet algorithme.

In [12]:
def graphe(P,T,pre,post,m):
    V=[]
    E=[]
    return(V,E)

## Exercice 3 — Model checking

**Question 1.** Écrivez une fonction qui vérifie si un RdP **borné** est **bloquant**, ainsi qu’une fonction qui, le cas échéant, renvoie un **marquage bloqué**.


In [13]:
def estbloquant(P,T,pre,post,m):
    # teste si un reseau est bloquant
    return False

def marquagebloque(P,T,pre,post,m):
    # renvoie un marquage bloqué
    return None


**Question 2.** Écrivez une fonction qui vérifie si un RdP **borné** est **propre** et, le cas échéant, renvoie un marquage accessible *M* à partir duquel il est **impossible de revenir** au marquage initial.


In [14]:
def estpropre(P,T,pre,post,m):
    # teste si un reseau est propre
    return True

**Question 3.** Écrivez une fonction qui vérifie si un réseau de Petri **borné** est **quasi-vivant** et, le cas échéant, renvoie une **transition** qui ne l’est pas.
Écrire une fonction qui teste si un réseau de Petri borné est quasi-vivant et qui renvoie une transition non-quasi-vivante le cas échéant.

In [15]:
def estqv(P,T,pre,post,m):
    # teste si un reseau est quasi-vivant
    return False

**Question 4.** Écrivez une fonction qui vérifie si un réseau de Petri **borné** est **vivant** et, le cas échéant, renvoie un **marquage accessible M** ainsi qu’une **transition** qui **n’est pas vivante** à partir de *M*.
Écrire une fonction qui teste si un réseau de Petri borné est vivant et qui renvoie un marquage accessible M et une transition non-vivante à partir de M le cas échéant.

In [16]:
def estvivant(P,T,pre,post,m):
    # teste si un reseau est vivant
    return False

## Exercice 4 — Traverser la rivière
Un groupe de 4 personnes (**A, B, C, D**) doivent traverser un pont mal éclairé pour franchir une rivière.  
Le pont ne supporte le poids que de **deux personnes**. Ils ont à disposition **une seule lampe de poche** (qui doit être allumée pour chaque traversée).  
Pour traverser le pont, A met **10 min**, B met **5 min**, C met **2 min** et D met **1 min**.

**Objectif.** Déterminer le **temps minimal** nécessaire pour que tout le groupe traverse le pont.

**Question 1.** Représentez ce problème sous la forme d’un **RdP** et en déduire le **temps minimal** de la traversée.


In [4]:
import heapq

# Personnes : A=0 (10min), B=1 (5min), C=2 (2min), D=3 (1min)
durees = [10, 5, 2, 1]
noms = ['A', 'B', 'C', 'D']

# Places :
# 0-3 : A,B,C,D a gauche
# 4-7 : A,B,C,D a droite
# 8   : torche a gauche
# 9   : torche a droite
P4 = 10

# Transitions :
# - 6 paires allant vers la droite (torche gauche -> droite)
# - 4 singles allant vers la droite
# - 4 singles revenant vers la gauche (torche droite -> gauche)
# - 6 paires revenant vers la gauche
pairs   = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
singles = [0,1,2,3]

transitions_list = []
durees_t = []

for (i,j) in pairs:     # paires vers la droite
    transitions_list.append(('right', i, j)); durees_t.append(max(durees[i], durees[j]))
for i in singles:        # singles vers la droite
    transitions_list.append(('right', i, None)); durees_t.append(durees[i])
for i in singles:        # singles vers la gauche
    transitions_list.append(('left', i, None)); durees_t.append(durees[i])
for (i,j) in pairs:     # paires vers la gauche
    transitions_list.append(('left', i, j)); durees_t.append(max(durees[i], durees[j]))

T4 = len(transitions_list)

# Construction des matrices pre et post
pre4  = [[0]*T4 for _ in range(P4)]
post4 = [[0]*T4 for _ in range(P4)]

for t_idx, (direction, i, j) in enumerate(transitions_list):
    if direction == 'right':
        # personnes quittent la gauche, torche quitte la gauche
        pre4[i][t_idx]    = 1;  pre4[8][t_idx]    = 1
        post4[i+4][t_idx] = 1;  post4[9][t_idx]   = 1
        if j is not None:
            pre4[j][t_idx]    = 1
            post4[j+4][t_idx] = 1
    else:  # 'left'
        # personnes quittent la droite, torche quitte la droite
        pre4[i+4][t_idx]  = 1;  pre4[9][t_idx]    = 1
        post4[i][t_idx]   = 1;  post4[8][t_idx]   = 1
        if j is not None:
            pre4[j+4][t_idx] = 1
            post4[j][t_idx]  = 1

# Marquage initial : tout le monde a gauche, torche a gauche
m4      = [1,1,1,1,0,0,0,0,1,0]
# Marquage but : tout le monde a droite, torche a droite
m4_goal = [0,0,0,0,1,1,1,1,0,1]

# Dijkstra pour trouver le chemin de temps minimal
def temps_min(P, T, pre, post, m_init, m_goal, durees_t):
    m_init_t = tuple(m_init)
    m_goal_t = tuple(m_goal)

    dist = {m_init_t: 0}
    prev = {m_init_t: (None, None)}   # m_precedent, transition utilisee
    heap = [(0, m_init_t)]

    while heap:
        d, m_t = heapq.heappop(heap)

        if m_t == m_goal_t:
            # Reconstruction du chemin
            path = []
            curr = m_t
            while prev[curr][0] is not None:
                path.append(prev[curr])
                curr = prev[curr][0]
            path.reverse()
            return d, path

        if d > dist[m_t]:
            continue

        m = list(m_t)
        for t in franchissable(P, T, pre, post, m):
            m_prime_t = tuple(apply(P, T, pre, post, m, t))
            new_d = d + durees_t[t]
            if m_prime_t not in dist or new_d < dist[m_prime_t]:
                dist[m_prime_t] = new_d
                prev[m_prime_t] = (m_t, t)
                heapq.heappush(heap, (new_d, m_prime_t))

    return None, None  # but non accessible

temps, chemin = temps_min(P4, T4, pre4, post4, m4, m4_goal, durees_t)

print(f"temps minimal : {temps} minutes\n")

for (m_from, t) in chemin:
    direction, i, j = transitions_list[t]
    fleche = '->' if direction == 'right' else '<-'
    if j is not None:
        print(f"  {noms[i]} + {noms[j]}  {fleche}  ({durees_t[t]} min)")
    else:
        print(f"  {noms[i]}  {fleche}  ({durees_t[t]} min)")

temps minimal : 17 minutes

  C + D  ->  (2 min)
  D  <-  (1 min)
  A + B  ->  (10 min)
  C  <-  (2 min)
  C + D  ->  (2 min)


## Exercice 5 — Protocole
Vous considérez un **protocole** de connexion/déconnexion entre un **client** et un **serveur**.

### Connexion
1. Le **client** initie la connexion en envoyant une **demande de connexion (DC)**, puis **attend**.
2. À la réception de **DC**, le **serveur** envoie **CC (confirmation de connexion)** puis passe dans l’**état connecté**.
3. Le **client** passe dans l’**état connecté** à la réception de **CC**.

### Déconnexion
1. Le **client (ou le serveur)** envoie une **demande de déconnexion** **DD1 (ou DD2)** puis **attend**.
2. À la réception de **DD1 (ou DD2)**, le **serveur (ou le client)** envoie une **confirmation de déconnexion** **CD1 (ou CD2)**, puis passe dans l’**état déconnecté**.
3. À la réception de **CD1 (ou CD2)**, le **client (ou le serveur)** passe dans l’**état déconnecté**.

**Question 1.** Représentez ce protocole à l’aide d’un **RdP**. Est-il **borné** ? **bloquant** ? Si oui, **justifiez**.

In [17]:
# Le réseau modélisant le système

P = 13
T = 9
pre = []
post = []
m = ()

print(estborne(P,T,pre,post,m))
print(estbloquant(P,T,pre,post,m))
print(marquagebloque(P,T,pre,post,m))


True
False
None
